# 04_model_comparison

Responsabilite:
- comparer les modeles testes sur les donnees maritimes pretraitees
- centraliser les scores et graphiques de comparaison

Entrees:
- donnees deja pretraitees depuis les notebooks ou `src/preprocessing.py`

Sorties:
- tableaux de scores, courbes et elements de comparaison de modeles

Execution:
- ouvrir le notebook depuis la racine du projet et executer les cellules


In [18]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder

# from pandas.plotting import scatter_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, make_scorer, recall_score


In [2]:
# === CONFIGURATION ===
pd.set_option("display.max_columns", None)

# === IMPORT DU FICHIER ===
source = '../data/frioul_if_reduit.csv'
df = pd.read_csv(source, index_col=0)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36516 entries, 0 to 63666
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   VentNoeud                36516 non-null  float64
 1   HouleDominante           36516 non-null  float64
 2   HouleMax                 36516 non-null  float64
 3   HoulePeriode             36516 non-null  float64
 4   Mer                      36516 non-null  int64  
 5   Temperature              36516 non-null  float64
 6   JourNuit                 36516 non-null  int64  
 7   VentDanger               36516 non-null  int64  
 8   HouleDanger              36516 non-null  int64  
 9   Ligne_Frioul-Vieux Port  36516 non-null  float64
 10  Ligne_IF-Frioul          36516 non-null  float64
 11  Ligne_IF-Vieux Port      36516 non-null  float64
 12  Ligne_Vieux Port-Frioul  36516 non-null  float64
 13  Ligne_Vieux Port-IF      36516 non-null  float64
 14  Annulation               36

In [7]:
X = df.drop(columns=['Annulation'])
y = df['Annulation']

# Séparation train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## Etape intermédiaire - Comparaison de diffférents modèles

In [44]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

In [48]:
for name, model in models.items():
    print(f"\n=== {name} ===")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


=== Logistic Regression ===
Accuracy: 0.9342825848849945
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      5673
           1       0.87      0.83      0.85      1631

    accuracy                           0.93      7304
   macro avg       0.91      0.90      0.90      7304
weighted avg       0.93      0.93      0.93      7304


=== Random Forest ===
Accuracy: 0.9346933187294633
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      5673
           1       0.88      0.82      0.85      1631

    accuracy                           0.93      7304
   macro avg       0.91      0.89      0.90      7304
weighted avg       0.93      0.93      0.93      7304


=== Decision Tree ===
Accuracy: 0.910460021905805
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      5673
           1       0.80      0.79      0.80      1631

    accuracy         

In [50]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append({"Modèle": name, "Accuracy": acc})

results_df = pd.DataFrame(results)
print(results_df.sort_values("Accuracy", ascending=False))


                Modèle  Accuracy
1        Random Forest  0.934419
0  Logistic Regression  0.934283
4                  KNN  0.920318
3                  SVM  0.919359
2        Decision Tree  0.909365


## Sélection de random forest et logisitc regression

#### optimisation des modèles pour maximiser le rappel ou le F1-score de la classe 1 (annulation)

In [16]:
# Réduit la taille de l’échantillon pour l’expérimentation
X_sample = X_train.sample(10000, random_state=42)
y_sample = y_train.loc[X_sample.index]
grid_rf.fit(X_sample, y_sample)


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 300, 500]},
             scoring='recall', verbose=2)

In [28]:
# Définition du scoring ciblé sur le rappel classe 1
scoring = make_scorer(recall_score, pos_label=1)


# Random Forest
param_grid_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': [None, 'balanced']
}
# param_grid_rf = {
#     'n_estimators': [100, 300, 500],
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'class_weight': [None, 'balanced']
# }
rf = RandomForestClassifier(random_state=42)


'''
utilise RandomizedSearchCV au lieu de GridSearchCV,  teste un nombre fixe de combinaisons aléatoires. C’est souvent suffisant
'''
# grid_rf = GridSearchCV(
#     estimator=rf,
#     param_grid=param_grid_rf,
#     scoring='recall',  # ou 'f1', ou 'recall' pour maximiser la détection des annulations
#     cv=5,
#     n_jobs=-1,
#     verbose=2
# )

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid_rf,
    n_iter=20,
    scoring=scoring,
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=42
)

# grid_rf.fit(X_train, y_train)

# print("Meilleurs paramètres :", grid_rf.best_params_)
# print("Meilleur rappel sur validation croisée :", grid_rf.best_score_)
random_search.fit(X_train, y_train)

print("Meilleurs paramètres :", random_search.best_params_)
print("Meilleur rappel sur validation croisée :", random_search.best_score_)




Fitting 3 folds for each of 20 candidates, totalling 60 fits
Meilleurs paramètres : {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 10, 'class_weight': 'balanced'}
Meilleur rappel sur validation croisée : 0.8680272252711424


In [30]:
# Logistic Regression

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],  # 'liblinear' gère l1 et l2
    'class_weight': [None, 'balanced']
}

lr = LogisticRegression(max_iter=1000, random_state=42)

grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    scoring='recall',  # ou 'f1'
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_lr.fit(X_train, y_train)

print("Logistic Regression - Meilleurs paramètres :", grid_lr.best_params_)
print("Logistic Regression - Meilleur rappel CV :", grid_lr.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Logistic Regression - Meilleurs paramètres : {'C': 0.01, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}
Logistic Regression - Meilleur rappel CV : 0.9207548832945491


- scoring='recall' : maximise la capacité du modèle à détecter les annulations (minimise les faux négatifs).

- scoring='f1' : équilibre entre rappel et précision pour la classe 1.

## Évaluation sur le jeu de test

In [32]:
# === Évaluation finale sur le jeu de test ===

# Récupère le meilleur modèle avec .best_estimator_ :
best_rf = grid_rf.best_estimator_
best_lr = grid_lr.best_estimator_

print("\n--- Évaluation Random Forest sur test ---")
y_pred_rf = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

print("\n--- Évaluation Logistic Regression sur test ---")
y_pred_lr = best_lr.predict(X_test)
print(classification_report(y_test, y_pred_lr))



--- Évaluation Random Forest sur test ---
              precision    recall  f1-score   support

           0       0.96      0.95      0.96      5673
           1       0.83      0.86      0.85      1631

    accuracy                           0.93      7304
   macro avg       0.90      0.91      0.90      7304
weighted avg       0.93      0.93      0.93      7304


--- Évaluation Logistic Regression sur test ---
              precision    recall  f1-score   support

           0       0.97      0.85      0.91      5673
           1       0.64      0.92      0.76      1631

    accuracy                           0.87      7304
   macro avg       0.81      0.89      0.83      7304
weighted avg       0.90      0.87      0.88      7304



Points importants
-  optimiationn sur le** rappe**l de la classe  (Annulation = 1)f.
- la validation croisée sur le train, p tu tstds les meilleurs modèles sur le jeu de test indépendant- C
Tu caimonpdres les rapports de classification pour choisir le modèle finratique.aut juste tester les deux meilleurs modèles sur le jeu de test et comparer leurs performances avant de choisir celui à déployer.

Si tu souhaites, je peux aussi te montrer comment faire une comparaison graphique des performances ou comment sauvegarder le modèle final.